# Week 3: Named Entity Extraction

This notebook explores the rule-based entity extractor for bedrooms, bathrooms, prices, square footage, and property amenities.

The extractor uses regular expressions for numeric entities and the Week 1 taxonomy for amenity matching. The cleaned remarks from Week 2 are used as input.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.entity_extractor import EntityExtractor

print("Project root:", project_root)

Considerations for entity extraction methods : the variations i used for the word varied more than doing bathroom or bedroom - i had to look for other wordings - I assumed price would need additional literals - like priced at, cost, listed at, but it was already written.



## Load the Week 3 data

In [ ]:
cleaned_df = pd.read_csv(
    project_root / "data" / "processed" / "listing_sample_cleaned.csv"
)

labels_df = pd.read_csv(
    project_root / "data" / "processed" / "entity_labeling_template.csv"
)

print("Cleaned listings:", len(cleaned_df))
print("Reviewed labels:", len(labels_df))
print("Columns:", labels_df.columns.tolist())

## Test the extractor

This example shows how an unstructured listing remark is converted into a structured dictionary.

In [ ]:
extractor = EntityExtractor()

example = (
    "This 3 bedroom home has 2.5 bathrooms, 2000 square feet, "
    "and a community pool. The price is 850000."
)

print(extractor.extract_all(example))

## Examples from the real dataset

In [ ]:
for number, text in enumerate(
    cleaned_df["cleaned_remarks"].head(5),
    start=1,
):
    result = extractor.extract_all(text)
    print(f"\n--- LISTING {number} ---")
    print("Bedrooms:", result["bedrooms"])
    print("Bathrooms:", result["bathrooms"])
    print("Price:", result["price"])
    print("Square feet:", result["sqft"])
    print("Amenities found:", len(result["amenities"]))

## Labeled entity spans

The labeled dataset contains the starting and ending character positions of each entity found in 200 remarks.

In [ ]:
span_file = (
    project_root / "data" / "processed" / "labeled_entities.jsonl"
)

records = []
label_counts = Counter()

with open(span_file, encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        records.append(record)

        for entity in record["entities"]:
            start = entity["start"]
            end = entity["end"]
            assert record["text"][start:end] == entity["text"]
            label_counts[entity["label"]] += 1

print("Records:", len(records))
print("Entity spans:", sum(label_counts.values()))
print("Labels:", dict(label_counts))

In [ ]:
print("Remark preview:")
print(records[0]["text"][:500])

print("\nEntity spans:")
for entity in records[0]["entities"][:10]:
    print(entity)

## Evaluation results

Precision measures how often extracted entities are correct. Recall measures how many expected entities are found. F1 balances both measurements.

In [ ]:
metrics_file = project_root / "reports" / "week3_metrics.json"

with open(metrics_file, encoding="utf-8") as file:
    metrics = json.load(file)

metrics_table = pd.DataFrame(metrics).T
metrics_table[["precision", "recall", "f1"]] = (
    metrics_table[["precision", "recall", "f1"]] * 100
)

metrics_table

In [ ]:
overall_f1 = metrics["overall"]["f1"]

print(f"Overall F1: {overall_f1:.2%}")
print("Meets 85% target:", overall_f1 >= 0.85)

## Error analysis

The main errors came from written number words, hyphenated or fractional bathroom counts, and large numbers being mistaken for prices. Square-footage extraction also confused living area with lot, deck, balcony, and garage measurements. Multi-unit properties were difficult because they could contain several valid bedroom and bathroom counts.

The overall score was strongly influenced by exact taxonomy matching. Bathroom and price extraction were less reliable than the overall score suggests.

## Conclusion

The rule-based extractor successfully converted listing remarks into structured entities and exceeded the 85% overall F1 target. Regex worked well for common numeric formats, while the Week 1 taxonomy provided direct amenity matching. The error analysis shows where additional patterns or a trained model could improve future results.